# Full-map curriculum with rankings from failed rollouts

Start with raw latent MSE and an empty local replay pool. Use 12 fixed goals across both rooms, eight expanding square bands, and three collection and fitting rounds per band. Each round samples up to 36 starts per goal, for up to 108 per goal per band. One-macro MPC collects fresh rollouts. The successful-pair pipeline still removes loops and uses the same temporal ranking rule on eligible consecutive simulator frames.

Each failed rollout contributes up to five distinct, evenly spaced eligible observations. Compare each one to the eligible frames of all successful rollouts to the same goal collected so far, including the current round. Use the incoming adapter for all comparisons, or raw I-JEPA MSE before the first fit. Keep the closest frame per successful rollout and average the three best scores from distinct rollouts. Generate a pair for each score difference greater than the tie tolerance. Lower scores rank better. If there are fewer than three usable successful rollouts for that goal, the failure contributes no pairs.

The usual goal-balanced sampler draws from the cumulative pool with no failure quota. Failure rankings enter this pool as generated. The frozen model checkpoints and compact reports live on Drive. Raw rollouts, replay, and reference tokens stay on the Colab VM.

Select a GPU runtime and choose **Run all**. Each band displays its summary, a sample failed ranking with matches when available, and plots before the next band starts.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
# Leave blank to start a fresh run. Set a printed ID to resume within the same Colab VM.
RESUME_LOCAL_RUN_ID = ''
# Set a Drive run ID to redraw completed bands on CPU without running the curriculum.
PLOT_SAVED_RUN_ID = ''
STARTS_PER_GOAL = 36  # Fresh starts per goal in each round.
ROUNDS_PER_BAND = 3
LAST_BAND = 8
PLOT_GOALS = [1, 4, 5, 10]  # Set to list(range(12)) to see every collection goal.
SAVE_REPORTS_TO_DRIVE = True

RUN_ID = RESUME_LOCAL_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = Path('/content/experience_mpc_route_feedback_full_runs') / RUN_ID
REPORT_ROOT = DRIVE / 'datasets/curriculum_cost_adapter/experience_mpc_route_feedback_full_square_v1'
REPORTS = REPORT_ROOT / RUN_ID
if not PLOT_SAVED_RUN_ID and RESUME_LOCAL_RUN_ID and not (OUTPUT/'experiment.json').is_file():
    raise FileNotFoundError(f'No local run at {OUTPUT}. Resuming requires the same Colab VM.')
REPO=Path('/content/i-jepa-wm-experience-route-feedback-full')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='e7385fec8e36a2d1eb09dc961c0ed1a41648c7ec'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

if PLOT_SAVED_RUN_ID:
    print('Reading saved reports:', REPORT_ROOT / PLOT_SAVED_RUN_ID)
else:
    print('Run ID:', RUN_ID)
    print('Local run:', OUTPUT)
    if SAVE_REPORTS_TO_DRIVE:
        print('Reports on Drive:', REPORTS)

## Set up the frozen encoder, world model and renderer

In [ ]:
if not PLOT_SAVED_RUN_ID:
    assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
    print('GPU:', torch.cuda.get_device_name())
    if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
    else:
        assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
        assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
        shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
    shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
    shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
    if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
    assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
    shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
    ARTIFACTS.mkdir(parents=True,exist_ok=True)
    for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
        assert source.is_file(), f'Missing frozen artifact: {source}'
        if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
        assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band):
    return [sys.executable, '-m', 'src.mylonas_ijepa_wm.experience_square', operation,
            '--output-dir', OUTPUT, '--world-model-checkpoint', CHECKPOINT,
            '--encoder-checkpoint', ENCODER, '--dino-wm-root', DINO_ROOT, '--device', 'cuda',
            '--starts-per-goal', STARTS_PER_GOAL, '--rounds-per-band', ROUNDS_PER_BAND,
            '--full-route-feedback', '--band', band]

## Plot and summarize a completed band

The trajectories are overlaid on the incoming cost surface used to collect that round. The last panel shows the cost surface after the band. Panels for a given goal and band share one absolute MSE scale. The amber square edges show where starts were sampled. Orange crosses mark erased loop observations, and white dots mark retained training observations. Magenta dots mark sampled failure observations. The table shows how many failure pairs were generated and how often the optimizer drew them.

The lower row shows successful paths after loop filtering on the same incoming heatmaps. Green points mark all observations that survive loop removal. The lines stop at gaps where observations were erased. The white dots above still mark the smaller set used in ranking pairs.

To redraw a completed run, set `PLOT_SAVED_RUN_ID` to its Drive report ID and choose **Run all**. This mode reads the saved comparison reports and works on a CPU runtime.

The route table distinguishes reaching the opposite room from reaching the goal. A rollout can cross the doorway and still fail.

In [ ]:
from IPython.display import Image, Markdown, display
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def read_json(path):
    return json.loads(path.read_text())


def wall_overlay(ax):
    for y,height in ((4,22),(34,26)):
        ax.add_patch(Rectangle((29,y),6,height,facecolor='#555',alpha=.75,zorder=2))

def square_edges(ax, goal, band):
    gx,gy=goal
    for radius,style in zip(band,('-','--')):
        if radius:
            ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,fill=False,
                                   edgecolor='#db9123',linestyle=style,linewidth=1.2,zorder=3))


def after_loop_filter(row):
    path = np.asarray(row['boundary']['positions'], dtype=float)
    path[row.get('erased_steps', [])] = np.nan
    return path


def draw_band(band, manifest, report, plot_dir):
    axis = report['axis']
    shape = (len(axis), len(axis))
    checkpoints = {(row['band'], row['round']): row for row in report['stages']}
    rounds = manifest['rounds_per_band']
    current = [checkpoints[band, n] for n in range(1, rounds + 1)]
    active = sorted({row['case']['goal_index'] for stage in current for row in stage['collection']})
    if not active:
        return []
    selected = [i for i in PLOT_GOALS if i in active] or active[:1]
    saved = []
    for goal_index in selected:
        goal = manifest['goals'][goal_index]
        groups = [[row for row in stage['collection'] if row['case']['goal_index'] == goal_index]
                  for stage in current]
        fields = [report['collection_raw_grid'][goal_index] if band == 1
                  else checkpoints[band - 1, rounds]['collection_grid'][goal_index]]
        fields += [stage['collection_grid'][goal_index] for stage in current]
        fields = [np.asarray(values, dtype=float).reshape(shape) for values in fields]
        vmax = max(max(float(np.nanmax(field)) for field in fields), 1e-9)
        fig, axes = plt.subplots(2, rounds + 1, figsize=(4.0 * (rounds + 1), 8.6), layout='constrained')
        for i, (ax, field) in enumerate(zip(axes[0], fields)):
            mesh = ax.pcolormesh(axis, axis, field, shading='nearest', cmap='viridis', vmin=0, vmax=vmax)
            wall_overlay(ax)
            square_edges(ax, goal, manifest['bands'][band - 1])
            if i < rounds:
                episodes = groups[i]
                for row in sorted(episodes, key=lambda r: r['success']):
                    path = np.asarray(row['boundary']['positions'])
                    success = row['success']
                    color = '#64f39a' if success else '#ff727a'
                    shown = path if success else path[:6]
                    ax.plot(*shown.T, color=color, alpha=.85, linewidth=1.5, zorder=4)
                    if not success:
                        ax.plot(*path[[0, -1]].T, color=color, ls='--', alpha=.6, zorder=4)
                    erased = row.get('erased_steps', [])
                    if erased:
                        ax.scatter(*path[erased].T, s=12, marker='x', c='#ffad68', alpha=.75, zorder=5)
                    shown_points = row['retained_positions'] if success else row.get('route_feedback', {}).get('positions', [])
                    points = np.asarray(shown_points).reshape(-1, 2)
                    if len(points):
                        ax.scatter(*points.T, s=17,
                                   facecolors='white' if success else '#f489fc',
                                   edgecolors='#252526', linewidths=.55, zorder=6)
                    ax.scatter(*path[min(5, len(path) - 1)], s=27, marker='D', color='#ffc341',
                               edgecolor='#252526', linewidths=.5, zorder=7)
                    ax.scatter(*path[0], s=39, facecolors='none', edgecolors=color, zorder=8)
                    ax.scatter(*path[-1], s=37, color=color, marker='x', zorder=8)
                count = sum(row['success'] for row in episodes)
                label = 'Raw MSE' if band == 1 and i == 0 else 'Incoming cost'
                title = f'Round {i + 1} · {count}/{len(episodes)} success\n{label}'
            else:
                title = 'After the final fit' if current[-1]['updates'] else 'After the final round (no fit)'
            ax.scatter(*goal, marker='*', s=120, c='black', edgecolors='white', zorder=9)
            ax.set(title=title, xlim=(4, 60), ylim=(4, 60), aspect='equal', xlabel='Ball x')
        for i, ax in enumerate(axes[1, :-1]):
            mesh = ax.pcolormesh(axis, axis, fields[i], shading='nearest', cmap='viridis', vmin=0, vmax=vmax)
            wall_overlay(ax)
            square_edges(ax, goal, manifest['bands'][band - 1])
            successes = [row for row in groups[i] if row['success']]
            for row in successes:
                path = after_loop_filter(row)
                ax.plot(*path.T, color='#64f39a', alpha=.85, linewidth=1.5, marker='.',
                        markersize=3.5, zorder=4)
            erased_count = sum(len(row.get('erased_steps', [])) for row in successes)
            detail = f'{erased_count} observations erased' if successes else 'No successful rollouts'
            ax.scatter(*goal, marker='*', s=120, c='black', edgecolors='white', zorder=9)
            ax.set(title=f'Round {i + 1} after loop filtering\n{detail}',
                   xlim=(4, 60), ylim=(4, 60), aspect='equal', xlabel='Ball x')
        axes[1, -1].set_axis_off()
        axes[0, 0].set_ylabel('Ball y')
        axes[1, 0].set_ylabel('Ball y')
        fig.colorbar(mesh, ax=list(axes.flat), shrink=.65, label='Latent MSE')
        fig.suptitle(f'Band {band} · Collection goal {goal_index} ({goal[0]:.2f}, {goal[1]:.2f})')
        fig.legend(handles=[
            Line2D([], [], color='#1b984e', label='Successful path'),
            Line2D([], [], color='#c44048', label='Failed first five actions'),
            Line2D([], [], color='#c44048', ls='--', label='Failed start to end'),
            Line2D([], [], marker='o', mfc='white', mec='#252526', ls='none', label='Retained success frame'),
            Line2D([], [], marker='x', color='#e57b31', ls='none', label='Erased loop frame'),
            Line2D([], [], marker='o', mfc='#f489fc', mec='#252526', ls='none', label='Sampled failure observation'),
            Line2D([], [], marker='D', mfc='#ffc341', mec='#252526', ls='none', label='First macro endpoint'),
        ], loc='upper center', bbox_to_anchor=(.5, -.005), ncol=4, frameon=False)
        path = plot_dir / f'goal_{goal_index}_band_{band}.png'
        fig.savefig(path, dpi=125, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))
        saved.append(path)
    return saved


def show_band(band):
    manifest = read_json(OUTPUT/'experiment.json')['manifest']
    report_path = OUTPUT/f'comparison_band_{band}.json'
    report = read_json(report_path)
    rows = []
    example = None
    route_rows = []
    exports = [OUTPUT/'experiment.json', OUTPUT/'runtime.json', report_path]
    for n in range(1, ROUNDS_PER_BAND + 1):
        folder = OUTPUT/f'band_{band}_round_{n}'
        stage = read_json(folder/'complete.json')
        training = read_json(folder/'training.json')
        sampling = training.get('replay_sampling', {})
        episodes = stage['collection']
        route_rows.extend(dict(round=n, route=row['case']['route'], success=row['success'],
                               crossed=row['reached_other_room'], steps=row['steps'])
                          for row in episodes)
        scored = [row for row in episodes if row.get('route_feedback', {}).get('scores') is not None]
        if example is None and scored:
            example = scored[0]
        draws = sampling.get('total_draws', 0)
        failures = sampling.get('failure_draws', 0)
        rows.append(dict(round=n, attempts=len(episodes), successes=sum(r['success'] for r in episodes),
                         scored_failures=len(scored),
                         later_better_pairs=sum(a < b for r in scored for a, b in r['route_feedback']['pairs']),
                         earlier_better_pairs=sum(a > b for r in scored for a, b in r['route_feedback']['pairs']),
                         loop_episodes=sum(bool(r.get('erased_steps')) for r in episodes),
                         loop_frames_removed=sum(len(r.get('erased_steps', [])) for r in episodes),
                         eligible_pairs_removed_by_loop_filter=sum(r.get('loop_removed_pairs', 0) for r in episodes),
                         new_success_pairs=sum(r['retained_pairs'] for r in episodes if r['success']),
                         new_failure_pairs=sum(r['retained_pairs'] for r in episodes if not r['success']),
                         failure_draws=failures, total_draws=draws,
                         failure_draw_pct=round(100 * failures / draws, 2) if draws else None,
                         updates=stage['updates']))
        exports += [folder/'complete.json', folder/'training.json']
    final = folder/'adapter.pt'
    if final.is_file():
        exports.append(final)
    plot_dir = OUTPUT/'plots'
    plot_dir.mkdir(exist_ok=True)
    table = pd.DataFrame(rows).set_index('round')
    display(table)
    by_route = pd.DataFrame(route_rows).groupby(['round', 'route']).agg(
        attempts=('success', 'size'), successes=('success', 'sum'), crossings=('crossed', 'sum'),
        median_steps=('steps', 'median'))
    display(by_route)
    route_summary = plot_dir/f'band_{band}_by_route.csv'
    by_route.to_csv(route_summary)
    exports.append(route_summary)
    if example is not None:
        audit = example['route_feedback']
        print(f"Example {example['case']['case_id']} ordered from best to worst")
        display(pd.DataFrame([
            dict(step=step, position=position, score=score,
                 closest_frames=[f"{m['case_id']} step {m['step']}" for m in matches],
                 match_positions=[tuple(round(x, 2) for x in m['position']) for m in matches])
            for step, position, score, matches in
            zip(audit['steps'], audit['positions'], audit['scores'], audit['matches'])
        ]).sort_values('score'))
    summary = plot_dir/f'band_{band}_summary.csv'
    table.to_csv(summary)
    exports += [summary, *draw_band(band, manifest, report, plot_dir)]
    if SAVE_REPORTS_TO_DRIVE:
        for path in exports:
            dest = REPORTS/path.relative_to(OUTPUT)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, dest)

## Run the curriculum and display each band

In [ ]:
assert 1 <= LAST_BAND <= 8 and ROUNDS_PER_BAND >= 2
assert PLOT_GOALS and all(0 <= i < 12 for i in PLOT_GOALS)
if PLOT_SAVED_RUN_ID:
    saved = REPORT_ROOT / PLOT_SAVED_RUN_ID
    manifest = read_json(saved/'experiment.json')['manifest']
    plot_dir = OUTPUT/'redrawn_plots'
    plot_dir.mkdir(parents=True, exist_ok=True)
    bands = [(band, saved/f'comparison_band_{band}.json') for band in range(1, LAST_BAND + 1)]
    available = [(band, report) for band, report in bands if report.is_file()]
    if not available:
        raise FileNotFoundError(f'No completed band reports in {saved}')
    for band, report in available:
        display(Markdown(f'## Band {band}'))
        draw_band(band, manifest, read_json(report), plot_dir)
else:
    for band in range(1, LAST_BAND + 1):
        display(Markdown(f'## Band {band}'))
        started = time.monotonic()
        finished = all((OUTPUT/f'band_{band}_round_{n}'/'complete.json').is_file()
                       for n in range(1, ROUNDS_PER_BAND + 1))
        if not finished:
            shell(command('stage', band), cwd=REPO)
        if not (OUTPUT/f'comparison_band_{band}.json').is_file():
            shell(command('compare', band), cwd=REPO)
        show_band(band)
        print(f'Band {band} finished in {(time.monotonic() - started) / 60:.1f} minutes.', flush=True)